In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path().resolve().parents[1]
print(f"Project root: {ROOT}")

In [ ]:
# ---- 設定 ----------------------------------------------------------------
bank_id = 1
prior   = "normal"

results_dir = ROOT / "EXP015" / "results"
mfi_path    = ROOT / "data" / "3PL" / "results" / f"summary_3pl_{bank_id}_MFI.csv"

pattern = re.compile(rf"summary_3pl_{bank_id}_DQN_{prior}_gamma_(.+)\.csv")
gamma_files = {
    pattern.match(p.name).group(1): p
    for p in sorted(results_dir.glob(f"summary_3pl_{bank_id}_DQN_{prior}_gamma_*.csv"))
    if pattern.match(p.name)
}

print(f"Found gamma values: {list(gamma_files.keys())}")

mfi = pd.read_csv(mfi_path)
print(f"MFI loaded: {mfi_path}")

In [ ]:
out_dir = results_dir
out_dir.mkdir(parents=True, exist_ok=True)

for gamma, path in gamma_files.items():
    dqn = pd.read_csv(path)

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(mfi["step"], mfi["RMSE"], label="MFI", color="steelblue", linewidth=1.8)
    ax.plot(dqn["step"], dqn["RMSE"], label=f"DQN (gamma={gamma})", color="tomato", linewidth=1.8)

    ax.set_xlabel("Step", fontsize=13)
    ax.set_ylabel("RMSE", fontsize=13)
    ax.set_title(
        f"RMSE Comparison: MFI vs DQN\n"
        f"(3PL bank {bank_id}, gamma={gamma}, TD target masked)",
        fontsize=13,
    )
    ax.legend(fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_xlim(1, dqn["step"].max())

    plt.tight_layout()

    out_path = out_dir / f"rmse_comparison_3pl_{bank_id}_gamma_{gamma}.png"
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved to {out_path}")